# Tutorial: Layer 3 Candidate Market Construction


## 1. 목적

            3번 레이어는 **후보 시장 구축**이다.

            여기서 중요한 질문은 "누가 잘하냐"가 아니라 먼저 이것이다.

            > 실제로 SSG가 영입 가능한 시장 안에 있는가?

            ## 사용한 모델/방법

            - deterministic gate: KBO 규정, 샐러리, 로스터, 국적/아시아쿼터 조건
            - market realism composite: 40-man/active/DFA/FA/마이너 계약/부상 신호를 합친 현실성 점수
            - source-fill packet: 계약, 의료, 한국행 의사, 여권/국적, 뉴스 근거를 채우는 작업표
            - data coverage audit: 어떤 리그/데이터가 얼마나 확보됐는지 추적


## 2. 왜 모델 전에 gate가 필요한가

            외국인 영입 문제에서는 좋은 선수를 찾는 것보다, **너무 좋아서 못 데려오는 선수**와 **정보가 부족해서 위험한 선수**를 먼저 거르는 것이 중요하다.

            그래서 3번 레이어는 ML 모델이라기보다, 스카우팅 부서의 현실 필터를 데이터화한 단계에 가깝다.


In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = ROOT.parent

OUT = ROOT / "outputs" / "tables"

def read_table(filename):
    path = OUT / filename
    if not path.exists():
        print(f"[missing] {path}")
        return pd.DataFrame()
    return pd.read_csv(path)

def take_cols(df, cols, n=10):
    keep = [c for c in cols if c in df.columns]
    if not keep:
        return df.head(n)
    return df.loc[:, keep].head(n)

def count_by(df, cols):
    keep = [c for c in cols if c in df.columns]
    if not keep or df.empty:
        return pd.DataFrame()
    return df.groupby(keep, dropna=False).size().reset_index(name="rows").sort_values("rows", ascending=False)

def assert_candidate_names_locked(df):
    sensitive_cols = {"player_name", "search_name", "team_or_org", "player_id"}
    exposed = sensitive_cols.intersection(df.columns)
    if exposed:
        print(f"Candidate-sensitive columns exist but are not displayed here: {sorted(exposed)}")


In [ ]:
gate = read_table("recruitment_gate_status_v33.csv")
take_cols(gate[gate["gate"].eq("G3")], ["gate", "layer", "progress_pct", "status", "decision", "blocking_gap"])


In [ ]:
coverage = read_table("data_coverage_by_league_v1.csv")
take_cols(coverage.sort_values("rows", ascending=False), ["league_bucket", "files", "size_mb", "rows", "detail"], n=12)


## 3. 시장별 확보 상태

            아래는 후보 시장을 어느 정도 채웠는지 보는 표다. 이 단계에서 후보명은 보지 않고, 슬롯/시장/상태 단위로만 본다.


In [ ]:
market_coverage = read_table("candidate_market_coverage_v0_3.csv")
take_cols(
    market_coverage,
    ["market_layer", "slot", "market_scope", "rows", "secured_rows", "research_lead_rows", "market_watch_rows", "medical_hold_rows", "data_status", "blocking_gap"],
    n=12,
)


In [ ]:
packet = read_table("ssg_fit_source_fill_packet_v0_1.csv")
assert_candidate_names_locked(packet)
slot_status = count_by(packet, ["fit_slot", "market_realism_status"])
slot_status.head(20)


## 4. source-fill readiness

            3번의 최종 산출물은 "최종 후보"가 아니라, 어떤 후보군에 어떤 추가 근거가 필요한지 알려주는 작업표다.

            대표적으로 필요한 수동 확인:

            - 계약/바이아웃/옵트아웃
            - 메디컬/부상 이력
            - 한국행 의사
            - 국적/아시아쿼터 조건
            - 기사/현지 소스 신뢰도


In [ ]:
readiness = count_by(packet, ["fit_slot", "source_fill_readiness_bucket"])
readiness.head(30)


In [ ]:
manual = count_by(packet, ["fit_slot", "manual_feasibility_priority_tier"])
manual.head(30)


## 5. 발표용 한 줄

            3번 레이어는 후보 시장을 "스탯 좋은 선수 목록"에서 "실제로 데려올 수 있는 선수 검토판"으로 바꾼 단계다.

            ## 연습문제

            source_fill_readiness_bucket이 낮은 슬롯을 하나 고르고, 그 슬롯에서 가장 먼저 채워야 할 정보가 무엇인지 적어보자.
